In [1]:
"""
Flow Logs - Gold Layer: Daily Traffic Summary
Aggregates flow data by day for trend analysis
"""

from pyspark.sql import functions as F

# ============================================================================
# CONFIGURATION
# ============================================================================

SILVER_PATH = "/Volumes/gitrepo/default/git_oci_aidp_silver/flow_logs/data"
GOLD_TABLE = "gitrepo.default.gold_flow_daily_summary"

print("=" * 70)
print("FLOW LOGS - GOLD LAYER: DAILY TRAFFIC SUMMARY")
print("=" * 70)

# ============================================================================
# READ SILVER
# ============================================================================

print("\n[STEP 1] Reading silver layer...")

silver_df = spark.read.parquet(SILVER_PATH)
record_count = silver_df.count()
print(f"✓ Loaded {record_count:,} records")

# ============================================================================
# BUILD DAILY SUMMARY
# ============================================================================

print("\n[STEP 2] Building daily summary...")

daily_summary = (
    silver_df
    .withColumn("flow_date", F.to_date("event_time"))
    .groupBy("flow_date")
    .agg(
        # Volume metrics
        F.count("*").alias("total_flows"),
        F.sum("bytes").alias("total_bytes"),
        F.sum("packets").alias("total_packets"),
        
        # Unique counts
        F.countDistinct("src_ip").alias("unique_src_ips"),
        F.countDistinct("dst_ip").alias("unique_dst_ips"),
        F.countDistinct("dst_port").alias("unique_dst_ports"),
        F.countDistinct(F.concat("src_ip", F.lit(":"), "dst_ip")).alias("unique_ip_pairs"),
        
        # Action breakdown
        F.sum(F.when(F.col("action") == "ACCEPT", 1).otherwise(0)).alias("accepted_flows"),
        F.sum(F.when(F.col("action") == "REJECT", 1).otherwise(0)).alias("rejected_flows"),
        
        # Direction breakdown
        F.sum(F.when(F.col("is_internal") == True, 1).otherwise(0)).alias("internal_src_flows"),
        F.sum(F.when(F.col("is_internal") == False, 1).otherwise(0)).alias("external_src_flows"),
        
        # Protocol breakdown
        F.sum(F.when(F.col("protocol_name") == "TCP", 1).otherwise(0)).alias("tcp_flows"),
        F.sum(F.when(F.col("protocol_name") == "UDP", 1).otherwise(0)).alias("udp_flows"),
        F.sum(F.when(F.col("protocol_name") == "ICMP", 1).otherwise(0)).alias("icmp_flows"),
        
        # Traffic stats
        F.avg("bytes").alias("avg_bytes_per_flow"),
        F.avg("packets").alias("avg_packets_per_flow"),
        F.avg("duration_seconds").alias("avg_duration_seconds"),
        F.max("bytes").alias("max_bytes_single_flow"),
        
        # Time range covered
        F.min("event_time").alias("first_event"),
        F.max("event_time").alias("last_event")
    )
    # Calculate rates
    .withColumn("reject_rate", 
        F.round(F.col("rejected_flows") / F.col("total_flows") * 100, 2))
    .withColumn("internal_rate",
        F.round(F.col("internal_src_flows") / F.col("total_flows") * 100, 2))
    .withColumn("total_gb",
        F.round(F.col("total_bytes") / (1024 * 1024 * 1024), 3))
    # Metadata
    .withColumn("processed_at", F.current_timestamp())
    .orderBy("flow_date")
)

daily_count = daily_summary.count()
print(f"✓ Aggregated into {daily_count} daily records")

# ============================================================================
# PREVIEW
# ============================================================================

print("\n[STEP 3] Preview...")

daily_summary.select(
    "flow_date",
    "total_flows",
    "total_gb",
    "unique_src_ips",
    "unique_dst_ips",
    "accepted_flows",
    "rejected_flows",
    "reject_rate"
).show(10, truncate=False)

# ============================================================================
# WRITE TO DELTA TABLE
# ============================================================================

print("\n[STEP 4] Writing to Delta table...")

(daily_summary.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(GOLD_TABLE))

print(f"✓ Written to {GOLD_TABLE}")

# ============================================================================
# VERIFY
# ============================================================================

print("\n[STEP 5] Verification...")

result_df = spark.table(GOLD_TABLE)
print(f"✓ Table record count: {result_df.count()}")
print(f"✓ Columns: {len(result_df.columns)}")

# Summary stats
print("\n" + "-" * 70)
print("SUMMARY STATISTICS")
print("-" * 70)

stats = result_df.agg(
    F.sum("total_flows").alias("total_flows"),
    F.sum("total_bytes").alias("total_bytes"),
    F.sum("rejected_flows").alias("total_rejected"),
    F.avg("reject_rate").alias("avg_reject_rate"),
    F.min("flow_date").alias("min_date"),
    F.max("flow_date").alias("max_date")
).collect()[0]

print(f"  Date Range: {stats['min_date']} to {stats['max_date']}")
print(f"  Total Flows: {stats['total_flows']:,}")
print(f"  Total Bytes: {stats['total_bytes']:,} ({stats['total_bytes']/(1024**4):.2f} TB)")
print(f"  Total Rejected: {stats['total_rejected']:,}")
print(f"  Avg Daily Reject Rate: {stats['avg_reject_rate']:.2f}%")

print("\n" + "=" * 70)
print("✓ GOLD DAILY SUMMARY COMPLETE")
print("=" * 70)

FLOW LOGS - GOLD LAYER: DAILY TRAFFIC SUMMARY

[STEP 1] Reading silver layer...


✓ Loaded 161,667,060 records

[STEP 2] Building daily summary...


✓ Aggregated into 32 daily records

[STEP 3] Preview...


+----------+-----------+--------+--------------+--------------+--------------+--------------+-----------+
|flow_date |total_flows|total_gb|unique_src_ips|unique_dst_ips|accepted_flows|rejected_flows|reject_rate|
+----------+-----------+--------+--------------+--------------+--------------+--------------+-----------+
|2025-12-18|1001002    |0.44    |9482          |3758          |586463        |412056        |41.16      |
|2025-12-19|5050056    |1.945   |19687         |7127          |3489275       |1544102       |30.58      |
|2025-12-20|4788529    |1.401   |20827         |8894          |3391567       |1380087       |28.82      |
|2025-12-21|4791066    |1.183   |19663         |6830          |3396044       |1378302       |28.77      |
|2025-12-22|5001449    |1.771   |19909         |7002          |3393300       |1591013       |31.81      |
|2025-12-23|4835465    |1.404   |17095         |4991          |3303246       |1516411       |31.36      |
|2025-12-24|5090529    |1.46    |17586        

✓ Written to gitrepo.default.gold_flow_daily_summary

[STEP 5] Verification...


✓ Table record count: 32
✓ Columns: 25

----------------------------------------------------------------------
SUMMARY STATISTICS
----------------------------------------------------------------------


  Date Range: 2025-12-18 to 2026-01-18
  Total Flows: 161,667,060
  Total Bytes: 48,332,414,324 (0.04 TB)
  Total Rejected: 47,564,766
  Avg Daily Reject Rate: 29.90%

✓ GOLD DAILY SUMMARY COMPLETE


In [2]:
%sql
-- Key metrics overview
SELECT 
    flow_date,
    total_flows,
    total_gb,
    unique_src_ips,
    unique_dst_ips,
    reject_rate,
    internal_rate
FROM gitrepo.default.gold_flow_daily_summary
ORDER BY flow_date;

In [4]:
%sql
-- High reject rate days (potential attack days)
SELECT 
    flow_date,
    total_flows,
    rejected_flows,
    reject_rate,
    unique_src_ips
FROM gitrepo.default.gold_flow_daily_summary
WHERE reject_rate > 35
ORDER BY reject_rate DESC;


In [7]:
%sql
-- Traffic volume trends
SELECT 
    flow_date,
    total_gb,
    total_flows,
    avg_bytes_per_flow,
    max_bytes_single_flow
FROM gitrepo.default.gold_flow_daily_summary
ORDER BY total_gb DESC;

In [ ]:
select * from 

In [6]:
%sql
-- Protocol breakdown by day
SELECT 
    flow_date,
    tcp_flows,
    udp_flows,
    icmp_flows,
    ROUND(tcp_flows * 100.0 / total_flows, 1) AS tcp_pct,
    ROUND(udp_flows * 100.0 / total_flows, 1) AS udp_pct
FROM gitrepo.default.gold_flow_daily_summary
ORDER BY flow_date;